In [1]:
import pandas as pd

In [4]:
df = pd.read_csv("employee_attrition.csv")

In [ ]:
df.head()

In [ ]:
df.Attrition.value_counts(normalize=True)

In [5]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import OneHotEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping



# ===============================
# 3. Basic Cleaning
# ===============================
# Convert target to binary
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Drop unnecessary columns if present
drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=[col for col in drop_cols if col in df.columns])


# ===============================
# 4. Feature / Target Split
# ===============================
X = df.drop("Attrition", axis=1)
y = df["Attrition"]


# ===============================
# 5. Identify Numerical & Categorical Columns
# ===============================
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns


# ===============================
# 6. Preprocessing Pipeline
# ===============================
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)


# ===============================
# 7. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ===============================
# 8. Build Deep Learning Model
# ===============================
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# ===============================
# 9. Early Stopping
# ===============================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


# ===============================
# 10. Train Model
# ===============================
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    class_weight={0:1, 1:2},   # handles class imbalance
    verbose=1
)


# ===============================
# 11. Evaluation
# ===============================
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_prob))


# ===============================
# 12. Save Model
# ===============================
model.save("attrition_dl_model.h5")


C:\Users\kruti\AppData\Local\Temp\ipykernel_21448\2890473878.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns
D:\new_kernel\myenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.3936 - loss: 0.9585 - val_accuracy: 0.7373 - val_loss: 0.6345
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6947 - loss: 0.6806 - val_accuracy: 0.8347 - val_loss: 0.5512
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8053 - loss: 0.6097 - val_accuracy: 0.8517 - val_loss: 0.4986
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8255 - loss: 0.5471 - val_accuracy: 0.8644 - val_loss: 0.4573
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8553 - loss: 0.5086 - val_accuracy: 0.8644 - val_loss: 0.4249
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8606 - loss: 0.5086 - val_accuracy: 0.8729 - val_loss: 0.3922
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8787 - loss: 0.4618 - val_accuracy: 0.8771 - val_loss: 0.3725
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8745 - loss: 0.4496 - val_accuracy: 0


Confusion Matrix:
[[236  11]
 [ 31  16]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.96      0.92       247
           1       0.59      0.34      0.43        47

    accuracy                           0.86       294
   macro avg       0.74      0.65      0.68       294
weighted avg       0.84      0.86      0.84       294

ROC-AUC Score: 0.7446808510638299


In [7]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("employee_attrition.csv")  # change path if needed


# ===============================
# 3. Basic Cleaning
# ===============================
# Convert target to binary
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Drop unnecessary columns if present
drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=[col for col in drop_cols if col in df.columns])


# ===============================
# 4. Feature / Target Split
# ===============================
X = df.drop("Attrition", axis=1)
y = df["Attrition"]


# ===============================
# 5. Train-Test Split (Before Preprocessing)
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ===============================
# 6. Identify Numerical & Categorical Columns
# ===============================
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns


# ===============================
# 7. Preprocessing Pipeline
# ===============================
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first', handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)


# ===============================
# 8. Logistic Regression Model
# ===============================
log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',   # handles imbalance automatically
    solver='liblinear'
)


# ===============================
# 9. Full Pipeline
# ===============================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', log_model)
])


# ===============================
# 10. Train Model
# ===============================
pipeline.fit(X_train, y_train)


# ===============================
# 11. Evaluation
# ===============================
y_pred = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_prob))


# ===============================
# 12. Save Model
# ===============================
import joblib
joblib.dump(pipeline, "attrition_logistic_model.pkl")


C:\Users\kruti\AppData\Local\Temp\ipykernel_21448\3593805434.py:54: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns



Confusion Matrix:
[[192  55]
 [ 15  32]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.78      0.85       247
           1       0.37      0.68      0.48        47

    accuracy                           0.76       294
   macro avg       0.65      0.73      0.66       294
weighted avg       0.84      0.76      0.79       294

ROC-AUC Score: 0.7985183909036094


['attrition_logistic_model.pkl']

In [8]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

from xgboost import XGBClassifier


# ===============================
# 2. Load & Prepare Data
# ===============================
df = pd.read_csv("employee_attrition.csv")

df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

X = df.drop("Attrition", axis=1)
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns


# ===============================
# 3. Preprocessing
# ===============================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)


# ===============================
# 4. XGBoost with L2 Regularization
# ===============================
xgb_l2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0,        # No L1
        reg_lambda=1,       # L2 regularization
        scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42
    ))
])

xgb_l2.fit(X_train, y_train)

l2_pred = xgb_l2.predict(X_test)
l2_prob = xgb_l2.predict_proba(X_test)[:, 1]

print("===== XGBoost (L2 Regularization) =====")
print(classification_report(y_test, l2_pred))
print("ROC-AUC:", roc_auc_score(y_test, l2_prob))


# ===============================
# 5. XGBoost with L1 Regularization
# ===============================
xgb_l1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=5,        # L1 regularization
        reg_lambda=0,
        scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42
    ))
])

xgb_l1.fit(X_train, y_train)

l1_pred = xgb_l1.predict(X_test)
l1_prob = xgb_l1.predict_proba(X_test)[:, 1]

print("\n===== XGBoost (L1 Regularization) =====")
print(classification_report(y_test, l1_pred))
print("ROC-AUC:", roc_auc_score(y_test, l1_prob))


C:\Users\kruti\AppData\Local\Temp\ipykernel_21448\958406320.py:37: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns
D:\new_kernel\myenv\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


===== XGBoost (L2 Regularization) =====
              precision    recall  f1-score   support

           0       0.88      0.92      0.90       247
           1       0.47      0.36      0.41        47

    accuracy                           0.83       294
   macro avg       0.68      0.64      0.66       294
weighted avg       0.82      0.83      0.82       294

ROC-AUC: 0.7791368765612886


D:\new_kernel\myenv\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:51:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



===== XGBoost (L1 Regularization) =====
              precision    recall  f1-score   support

           0       0.90      0.89      0.89       247
           1       0.45      0.47      0.46        47

    accuracy                           0.82       294
   macro avg       0.67      0.68      0.68       294
weighted avg       0.83      0.82      0.82       294

ROC-AUC: 0.7787061762425703


In [2]:
# ===============================
# 1. Imports
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from xgboost import XGBClassifier
import joblib


# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("employee_attrition.csv")

# Convert target to binary
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Drop unnecessary columns
drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=[col for col in drop_cols if col in df.columns])


# ===============================
# 3. Feature / Target Split
# ===============================
X = df.drop("Attrition", axis=1)
y = df["Attrition"]


# ===============================
# 4. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ===============================
# 5. Identify Column Types
# ===============================
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns


# ===============================
# 6. Preprocessing Pipeline
# ===============================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)


# ===============================
# 7. XGBoost Model
# ===============================
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    eval_metric='logloss',
    random_state=42
)


# ===============================
# 8. Full Pipeline
# ===============================
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])


# ===============================
# 9. Train Model
# ===============================
pipeline.fit(X_train, y_train)

# ===============================
# 10. Evaluation
# ===============================
y_pred = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_prob))


# ===============================
# 11. Save Model
# ===============================
joblib.dump(pipeline, "attrition_xgboost_model.pkl")


C:\Users\kruti\AppData\Local\Temp\ipykernel_21448\3106240714.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns



Confusion Matrix:
[[228  19]
 [ 30  17]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.92      0.90       247
           1       0.47      0.36      0.41        47

    accuracy                           0.83       294
   macro avg       0.68      0.64      0.66       294
weighted avg       0.82      0.83      0.82       294

ROC-AUC Score: 0.7791368765612886


['attrition_xgboost_model.pkl']